<div style="border-left: 20px solid #427255; padding: 24px 32px; background: #f4f6f8; margin-bottom: 24px;">

<p style="color:#2E75B6; font-size:13px; letter-spacing:3px; margin:0 0 8px 0; font-weight:bold;">STAKEHOLDER REPORT</p>

<h1 style="color:#1F3A5F; font-size:36px; margin:0 0 8px 0; line-height:1.1;">Predictive Funnel Analytics for E-Commerce</h1>

<p style="color:#555; font-size:18px; font-style:italic; margin:0 0 20px 0;">A GA4-Aligned Session Scoring &amp; Hurdle Model for Retail, Pricing, and E-Commerce Analytics</p>

<table style="border:none; font-size:14px; color:#333;">
<tr><td style="border:none; padding:2px 12px 2px 0; color:#555;"><b>Prepared by:</b></td><td style="border:none; padding:2px 0;">Troy Dela Rosa</td></tr>
<tr><td style="border:none; padding:2px 12px 2px 0; color:#555;"><b>Deliverable:</b></td><td style="border:none; padding:2px 0;">Stakeholder Summary — Two-Stage Hurdle Model (Propensity × Spend)</td></tr>
<tr><td style="border:none; padding:2px 12px 2px 0; color:#555;"><b>Audience:</b></td><td style="border:none; padding:2px 0;">Retail Analytics · Pricing · E-Commerce · Digital Merchandising Leadership</td></tr>
<tr><td style="border:none; padding:2px 12px 2px 0; color:#555;"><b>Scope:</b></td><td style="border:none; padding:2px 0;">100K customers · 2M+ events · GA4-equivalent schema</td></tr>
</table>

</div>

# Executive Summary

E-commerce funnels leak. Across the dataset analyzed, **94.8% of sessions** ended without a purchase — the familiar gap between add-to-cart and checkout that every retail analytics team sees in their GA4 reporting. This project builds a two-stage predictive system that scores every session in real time, estimates its expected revenue, and identifies the specific segments where a promotional nudge is both margin-safe and commercially productive.

The system is a **hurdle model**. Stage one is an XGBoost classifier that predicts the probability of conversion from session behaviour, user attributes, and campaign context — achieving a ROC-AUC of 0.80 with 3× lift in the top decile. Stage two estimates expected basket value using customer historical spend, because in-session behaviour turns out to predict *intent* but not *wallet*. Multiplying the two yields a decision-ready dollar figure per session that feeds pricing, promotion, bid optimization, and merchandising decisions.

<table style="border-collapse:collapse; width:100%; margin-top:16px;">
<tr>
<td style="border:none; padding:0 30px 0 0; width:33%;">
<div style="border-top:6px solid #b62ea449; border-left:2px solid #CCC; border-right:2px solid #CCC; border-bottom:2px solid #CCC; background:#F4F6F8; padding:16px 20px;">
<p style="color:#555; font-size:11px; letter-spacing:2px; font-weight:bold; margin:0 0 6px 0;">MODEL PERFORMANCE</p>
<p style="color:#1F3A5F; font-size:22px; font-weight:bold; margin:0;">ROC-AUC 0.80</p>
<p style="color:#555; font-size:13px; margin:4px 0 0 0;">3× top-decile lift</p>
</div>
</td>
<td style="border:none; padding:0 30px; width:33%;">
<div style="border-top:6px solid #3f2e7d42; border-left:2px solid #CCC; border-right:2px solid #CCC; border-bottom:2px solid #CCC; background:#F4F6F8; padding:16px 20px;">
<p style="color:#555; font-size:11px; letter-spacing:2px; font-weight:bold; margin:0 0 6px 0;">AT-RISK REVENUE</p>
<p style="color:#1F3A5F; font-size:22px; font-weight:bold; margin:0;">≈ $5.4M</p>
<p style="color:#555; font-size:13px; margin:4px 0 0 0;">Recoverable via targeted nudges</p>
</div>
</td>
<td style="border:none; padding:0 0 0 30px; width:33%;">
<div style="border-top:6px solid #00b87b1e; border-left:2px solid #CCC; border-right:2px solid #CCC; border-bottom:2px solid #CCC; background:#F4F6F8; padding:16px 20px;">
<p style="color:#555; font-size:11px; letter-spacing:2px; font-weight:bold; margin:0 0 6px 0;">FORECAST ACCURACY</p>
<p style="color:#1F3A5F; font-size:22px; font-weight:bold; margin:0;">≈ 13.5% error</p>
<p style="color:#555; font-size:13px; margin:4px 0 0 0;">After Platt calibration</p>
</div>
</td>
</tr>
</table>

# 1. Business Context

## The Problem

Retail and e-commerce teams routinely look at GA4 funnel reports and see the same pattern: large top-of-funnel traffic, strong add-to-cart rates, and a collapse at checkout. Three questions follow, and none of them is adequately answered by standard GA4 Exploration alone:

- **Which sessions will convert?** Pricing, inventory, and merchandising teams need a forward-looking demand signal, not a backward-looking conversion report.
- **How much revenue will a converting session generate?** Bidding, budget allocation, and CLV modelling require a dollar figure, not a probability.
- **Which non-converting sessions are worth a nudge?** Blanket discounting erodes margin; precision targeting on the right segment recovers revenue without training customers to wait for a coupon.

## Why GA4 Alignment Matters

The dataset used in this project was structured to **approximate a GA4 BigQuery export workflow**. `events.csv` is conceptually similar to GA4 `events_*` tables; `event_type` aligns with `event_name`; `customer_id` approximates `user_pseudo_id`; `session_id` approximates `ga_session_id`; and `purchase_amount` is comparable to `ecommerce.purchase_revenue`. While this project was not run on a live GA4 property, the modelling pipeline was designed to be transferable to GA4-style event data after validating event quality, session logic, identity mapping, revenue fields, and leakage safeguards.

## What This Report Covers

This document translates the modelling work into business terms for stakeholders outside the analytics function. It describes what was built, what the evidence says, what is actionable today, and what the known limits of the approach are. Full technical detail, model diagnostics, and calibration analysis are retained in the accompanying modelling notebook.

# 2. Approach

## Two-Stage Hurdle Model

Revenue data from e-commerce sessions is zero-inflated: most sessions generate no revenue, and a small number generate meaningful amounts. Modelling raw revenue directly blends two different problems and produces poor results. The standard remedy in retail analytics is a **hurdle model** — separate the two questions, then combine.

<table style="border-collapse:collapse; width:100%; margin-top:12px;">
<thead>
<tr style="background:#1F3A5F; color:white;">
<th style="padding:10px 14px; text-align:left; border:1px solid #1F3A5F;">Stage</th>
<th style="padding:10px 14px; text-align:left; border:1px solid #1F3A5F;">Model</th>
<th style="padding:10px 14px; text-align:left; border:1px solid #1F3A5F;">Question Answered</th>
</tr>
</thead>
<tbody>
<tr style="background:#F4F6F8;">
<td style="padding:10px 14px; border:1px solid #CCC;"><b>Stage 1</b></td>
<td style="padding:10px 14px; border:1px solid #CCC;">XGBoost Classifier (propensity)</td>
<td style="padding:10px 14px; border:1px solid #CCC;">Will this session convert?</td>
</tr>
<tr>
<td style="padding:10px 14px; border:1px solid #CCC;"><b>Stage 2</b></td>
<td style="padding:10px 14px; border:1px solid #CCC;">Segment-Based Spend Estimator (customer historical average)</td>
<td style="padding:10px 14px; border:1px solid #CCC;">If it converts, how much will it spend?</td>
</tr>
<tr style="background:#F4F6F8;">
<td style="padding:10px 14px; border:1px solid #CCC;"><b>Combined</b></td>
<td style="padding:10px 14px; border:1px solid #CCC;"><b>Expected Revenue = P(conversion) × Expected Spend</b></td>
<td style="padding:10px 14px; border:1px solid #CCC;">What is this session worth, in dollars?</td>
</tr>
</tbody>
</table>

## Data and Features

The pipeline aggregates raw events into session-level records, then attaches dimensional attributes from customers and campaigns. Three feature families survive the leakage audit and enter the model:

- **Behavioural (in-session):** session duration, event velocity, cart-to-view ratio, click-to-view ratio. These describe *how* the user behaved, not *what* they bought.
- **Customer:** age, loyalty tier, acquisition channel, device type.
- **Campaign:** channel, objective, expected uplift.

## Leakage Prevention

Behavioural funnel data carries a well-known leakage risk: features that look predictive often contain the answer. Four rules are enforced throughout:

1. Purchase-event counts are **never** used as features — they directly encode the target.
2. Historical spend is computed **time-aware** using `merge_asof`, so only transactions strictly *before* `session_start` are counted.
3. Imputers and encoders are **fit on the training split only**, then applied via `transform` to validation and test.
4. The 60/20/20 train/validation/test split is performed **before** any encoding or imputation.

> **The leakage journey is itself a finding.** An initial ROC-AUC of 1.00 was the first signal that features were leaking. Four iterations of feature removal brought the model to a realistic 0.80. This is a practice lesson with direct relevance to any live GA4 ML workflow.

# 3. Results

## Stage 1 — Propensity Model

The XGBoost classifier ranks sessions reliably: **ROC-AUC of 0.80** on the held-out test set, with the top decile converting at roughly three times the base rate. This is the lift that makes targeting economically sensible. Random Forest (balanced) and Logistic Regression were evaluated as baselines and consistently underperformed XGBoost, confirming the model choice.

## Stage 2 — Spend Estimator

A key honest finding: **in-session behavioural features do not predict spend amount** (R² ≈ −0.01). Clicks signal intent, not wallet. A segment-based estimator using customer historical average spend is both simpler and materially more accurate (**R² ≈ 0.65, MAE ≈ $28.57**). This is the estimator used in the production pipeline. In a live GA4 environment, this corresponds to building a user-property lookup of historical AOV and joining it at score time.

## Calibration — From Ranking to Forecasting

A model can rank well and still forecast badly. The uncalibrated classifier produced an expected-revenue sum roughly 8× larger than actual revenue — fine for sorting sessions, misleading for forecasting. **Platt scaling** on the validation set brought the calibrated forecast to within 13.5% of actual test-set revenue. The lesson for stakeholders: treat propensity scores as rankings until the model is calibrated, then treat them as probabilities.

<table style="border-collapse:collapse; width:100%; margin-top:12px;">
<thead>
<tr style="background:#1F3A5F; color:white;">
<th style="padding:10px 14px; text-align:left; border:1px solid #1F3A5F;">Metric</th>
<th style="padding:10px 14px; text-align:center; border:1px solid #1F3A5F;">Before Calibration</th>
<th style="padding:10px 14px; text-align:center; border:1px solid #1F3A5F;">After Platt Scaling</th>
</tr>
</thead>
<tbody>
<tr>
<td style="padding:10px 14px; border:1px solid #CCC;">Mean predicted conversion</td>
<td style="padding:10px 14px; border:1px solid #CCC; text-align:center;">Inflated</td>
<td style="padding:10px 14px; border:1px solid #CCC; text-align:center; color:#2E7D32;"><b>≈ actual 5.2%</b></td>
</tr>
<tr style="background:#F4F6F8;">
<td style="padding:10px 14px; border:1px solid #CCC;">Expected revenue (test set)</td>
<td style="padding:10px 14px; border:1px solid #CCC; text-align:center;">≈ $13.7M</td>
<td style="padding:10px 14px; border:1px solid #CCC; text-align:center; color:#2E7D32;"><b>≈ $1.97M vs actual $1.74M</b></td>
</tr>
<tr>
<td style="padding:10px 14px; border:1px solid #CCC;">Ranking (ROC-AUC)</td>
<td style="padding:10px 14px; border:1px solid #CCC; text-align:center;">0.80</td>
<td style="padding:10px 14px; border:1px solid #CCC; text-align:center;">0.80 (unchanged)</td>
</tr>
</tbody>
</table>

# 4. Business Value

The pipeline produces value across five core retail analytics functions. For each, the model provides a specific input, and the stakeholder team applies it to a specific decision.

<table style="border-collapse:collapse; width:100%; margin-top:12px;">
<thead>
<tr style="background:#1F3A5F; color:white;">
<th style="padding:10px 14px; text-align:left; border:1px solid #1F3A5F; width:22%;">Function</th>
<th style="padding:10px 14px; text-align:left; border:1px solid #1F3A5F; width:30%;">Model Input</th>
<th style="padding:10px 14px; text-align:left; border:1px solid #1F3A5F;">Business Application</th>
</tr>
</thead>
<tbody>
<tr style="background:#F4F6F8;">
<td style="padding:10px 14px; border:1px solid #CCC;"><b>Customer Demand Scoring</b></td>
<td style="padding:10px 14px; border:1px solid #CCC;">Session-level P(conversion)</td>
<td style="padding:10px 14px; border:1px solid #CCC;">Forward-looking demand signal for pricing and inventory. High-propensity traffic for a category is a leading indicator of demand.</td>
</tr>
<tr>
<td style="padding:10px 14px; border:1px solid #CCC;"><b>Promotion Targeting</b></td>
<td style="padding:10px 14px; border:1px solid #CCC;">At-risk segment (40–70% propensity)</td>
<td style="padding:10px 14px; border:1px solid #CCC;">Precision cart-recovery emails and targeted discounts for the segment that moves on a nudge — without eroding margin on buyers who would have converted anyway.</td>
</tr>
<tr style="background:#F4F6F8;">
<td style="padding:10px 14px; border:1px solid #CCC;"><b>Revenue Prioritization</b></td>
<td style="padding:10px 14px; border:1px solid #CCC;">Expected Revenue per session</td>
<td style="padding:10px 14px; border:1px solid #CCC;">Bid optimization in paid media, prioritized live-chat for high-value sessions, and channel-level budget allocation.</td>
</tr>
<tr>
<td style="padding:10px 14px; border:1px solid #CCC;"><b>Funnel Optimization</b></td>
<td style="padding:10px 14px; border:1px solid #CCC;">Drop-off diagnostics by channel / device / campaign</td>
<td style="padding:10px 14px; border:1px solid #CCC;">UX improvements, channel mix decisions, landing-page optimization — the GA4 Funnel Exploration view, enriched with propensity scores.</td>
</tr>
<tr style="background:#F4F6F8;">
<td style="padding:10px 14px; border:1px solid #CCC;"><b>Digital Merchandising</b></td>
<td style="padding:10px 14px; border:1px solid #CCC;">Behavioural signals + campaign attribution</td>
<td style="padding:10px 14px; border:1px solid #CCC;">Which products to feature, when to time promotions, which creative drives genuine intent versus vanity clicks.</td>
</tr>
</tbody>
</table>

## Segmentation for Action

The model separates sessions into three stakeholder-readable segments. The segment definitions are the practical output of the work — they translate a probability distribution into a marketing decision.

<table style="border-collapse:collapse; width:100%; margin-top:12px;">
<thead>
<tr style="background:#1F3A5F; color:white;">
<th style="padding:10px 14px; text-align:left; border:1px solid #1F3A5F;">Segment</th>
<th style="padding:10px 14px; text-align:center; border:1px solid #1F3A5F;">Propensity</th>
<th style="padding:10px 14px; text-align:left; border:1px solid #1F3A5F;">Recommended Action</th>
<th style="padding:10px 14px; text-align:left; border:1px solid #1F3A5F;">Rationale</th>
</tr>
</thead>
<tbody>
<tr>
<td style="padding:10px 14px; border:1px solid #CCC; color:#2E7D32;"><b>High-Certainty</b></td>
<td style="padding:10px 14px; border:1px solid #CCC; text-align:center;"><b>&gt; 80%</b></td>
<td style="padding:10px 14px; border:1px solid #CCC;">No intervention</td>
<td style="padding:10px 14px; border:1px solid #CCC;">Already converting. Discounts destroy margin on captive demand.</td>
</tr>
<tr style="background:#FFF8E1;">
<td style="padding:10px 14px; border:1px solid #CCC; color:#B88A00;"><b>At-Risk (target)</b></td>
<td style="padding:10px 14px; border:1px solid #CCC; text-align:center;"><b>40 – 70%</b></td>
<td style="padding:10px 14px; border:1px solid #CCC;">Cart-recovery email + targeted discount</td>
<td style="padding:10px 14px; border:1px solid #CCC;">Moderate intent but hesitating. A nudge converts. This is where incremental revenue lives.</td>
</tr>
<tr>
<td style="padding:10px 14px; border:1px solid #CCC; color:#555;"><b>Low Interest</b></td>
<td style="padding:10px 14px; border:1px solid #CCC; text-align:center;"><b>&lt; 40%</b></td>
<td style="padding:10px 14px; border:1px solid #CCC;">Brand awareness only</td>
<td style="padding:10px 14px; border:1px solid #CCC;">Conversion campaigns are ineffective here. Build the funnel; do not try to close it.</td>
</tr>
</tbody>
</table>

> **Bottom-line impact:** targeting the at-risk segment surfaces approximately **$5.4M** in recoverable revenue while protecting margin on high-certainty buyers. The top decile alone represents **~$1.57M** in incremental opportunity versus random targeting.

# 5. Recommendations

## Immediate (0–90 days)

- **Run an A/B test on the at-risk segment.** Validate which interventions (email, on-site pop-up, targeted discount) and incentive levels drive incremental conversions without eroding margin. Tie the experiment to the 40–70% propensity band defined by the model.
- **Deploy the model as a session-scoring service.** Package the saved artifacts (`xgb_classifier.pkl`, `customer_spend_lookup.csv`, `best_threshold.pkl`) behind a Flask or FastAPI endpoint. Trigger real-time marketing actions via Google Tag Manager or the CDP.
- **Stand up a monitoring dashboard.** Power BI or Looker Studio reporting on conversion rate by segment, model-drift indicators, and attributed revenue impact — shared with pricing, marketing, and merchandising stakeholders.

## Medium-Term (3–9 months)

- **Add product-level features.** Category, price band, premium flag. These feed the pricing role directly and let the model detect price sensitivity.
- **Add temporal features.** Day-of-week, hour-of-day, seasonality, and lag features on traffic volume.
- **Re-train on live GA4 outcomes.** Post-intervention data closes the feedback loop: the model learns from what the business actually did, not just what users did.

## Longer-Term

- **Move from conversion prediction to uplift modelling.** "Who will convert *because of* the intervention" is a different and more actionable question than "who will convert." It requires randomised-exposure data, which the A/B test above generates as a byproduct.
- **Integrate with the pricing engine.** Session-level demand scores feed dynamic pricing decisions — surge during high-propensity traffic, protect margin when low-intent traffic dominates.

# 6. Limitations and Caveats

Stakeholders should be aware of four honest limits on the current system.

- **Synthetic dataset.** The schema is 1:1 with GA4 and the pipeline is portable, but the numerical findings (ROC-AUC 0.80, $5.4M at-risk revenue, 13.5% calibration error) should be re-established on live data before being used for budget-level decisions.
- **Spend prediction is segment-based, not behavioural.** The model honestly reports that in-session signals do not predict basket size. Customers with no purchase history fall back to the global average. This is acceptable for most commerce decisions but limits accuracy for true first-time buyers.
- **Calibration is a snapshot.** Platt scaling was fit on the validation set at a single point in time. Probability calibration drifts as traffic composition shifts; the monitoring dashboard should track calibration, not only AUC.
- **Correlation, not causation.** The propensity model identifies *who will convert* — not *who will convert because of a nudge*. The A/B test in the immediate-actions list is what converts correlation into causal lift.

# 7. Closing

This project evolved from a supervised learning assignment into a retail analytics pipeline that delivers a decision-ready dollar value for every GA4 session. Three lessons carry forward regardless of which retailer the pipeline is deployed against:

- The features you **remove** matter as much as the ones you keep.
- The **questions** you ask matter more than the **algorithms** you run.
- The value of a model is measured in **business impact**, not metrics.

<div style="text-align:left; margin-top:40px; padding:24px; color:#555;">
<p style="font-style:italic; font-size:16px; margin:0 0 6px 0;">"The essence of strategy is choosing what not to do."</p>
<p style="font-size:13px; margin:0;">— Michael Porter</p>
</div>